## Libraries and Data.


In [81]:
import pandas as pd
import numpy as np
import string
import regex as re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords 
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

# Download NLTK data. Needs to be runned once per session. Can be commented after.
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to /home/siavashj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/siavashj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/siavashj/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [82]:
# Loading the data.

df_orig = pd.read_parquet('FNDDSeverything.parquet.gzip')
df = df_orig.select_dtypes(include="str")


## PIECE 1: Data Cleaning and Vectorizing


In [97]:
# print(re.sub(r',', ' ', "BACN,CAND,FL,4PC, Z"))

BACN CAND FL 4PC  Z


In [133]:
# df = df.dropna(subset=['description'])  # Remove nulls
# # df['description'] = df['description'].str.lower()  # lowercase

# ## Remove special characters.

# translator = str.maketrans('', '', string.punctuation)
# description_clean =[]
# for index, text in enumerate(df.description):
#     clean = text.translate(translator)
#     description_clean.append(clean)

# df['description_clean'] = description_clean

# df['description_clean'] =df['description_clean'].str.strip()  # remove whitespace

def preprocess_food_entries(text, Tokenize = True):
    """
    Complete preprocessing for food descriptions
    - Removes measurements and numbers
    - Tokenizes
    - Removes stopwords
    - Lemmatizes
    """
    sentence = ''

    # Step 1: Lowercase
    text = text.lower()

    # Step 0: replace ',' with ' '.
    text = re.sub(r',|/', ' ', text)
    

    # Step 3: Clean special characters but keep spaces
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Step 2: Remove all measurement patterns
    patterns_to_remove = [
        r'\d+\.?\d*\s?(?:oz|lb|lbs|g|kg|ml|l|cup|cups|tsp|tbsp|pt|qt|gal|mg|mcg|cal|count|ct|pack|pk|piece|pieces|pc|ounce|ounces|pound|pounds|gram|grams|kilogram|kilograms|milliliter|milliliters|liter|liters|teaspoon|teaspoons|tablespoon|tablespoons|pint|pints|quart|quarts|gallon|gallons|milligram|milligrams|microgram|micrograms|calorie|calories|fl|fluid)\b',
        r'\b\d+\.?\d*\b',  # standalone numbers
        r'\d+\.?\d*\s?%',  # percentages
        r'\d+\s?x\s?\d+',  # dimensions
    ]
    
    for pattern in patterns_to_remove:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    

    
    # Step 4: Clean multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 5: Tokenize
    tokens = word_tokenize(text)
    
    # Step 6: Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [t for t in tokens if t not in stop_words]
    
    # Step 7: Remove very short tokens
    tokens = [t for t in tokens if len(t) > 2]
    
    # Step 8: Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    tokens = list(dict.fromkeys(tokens))
    
    if Tokenize:
        return tokens
    else:    
        sentence = " ".join(tokens)
        return sentence

# Test

for sample in df.description[[0,24,81,124,134,550,551,552,553]]:
    tokens = preprocess_food_entries(sample)
    print(f"Original: {sample}")
    print(f"Tokens:   {tokens}")
    sent = preprocess_food_entries(sample, False)
    print(f"after token sent:   {sent}\n")

# # Test with edge cases
# edge_cases = [
#     "10oz organic tomatoes 2% milk",
#     "500ml coconut milk 100 calories",
#     "12x6 inch pizza 8 slices",
#     "2.5lb premium beef 15% fat",
#     "6pack 12oz cans soda"
# ]

# print("\n=== Edge Cases ===\n")
# for sample in edge_cases:
#     tokens = preprocess_food_descriptions(sample)
#     print(f"Original: {sample}")
#     print(f"Tokens:   {tokens}\n")


# # ## Remove very short descriptions (less than 2 words)
# # df['word_count'] = df['description_clean'].str.split().str.len()
# # df = df[df['word_count'] >= 2]
# # print(f"After removing short descriptions: {len(df)}")

# # Show sample
# print(df.description[24])
# print(df.description_clean[24])
# # print("\nSample cleaned descriptions:")
# # print(df[['description', 'description_clean']].head(10))




Original:  ADVANCEPIERRE FLAMEBROILED RIB SHAPED PORK PATTY TOPPED WITH BBQ SAUCE 
Tokens:   ['advancepierre', 'flamebroiled', 'rib', 'shaped', 'pork', 'patty', 'topped', 'bbq', 'sauce']
after token sent:   advancepierre flamebroiled rib shaped pork patty topped bbq sauce

Original:  DEATH WISH COFFEE CO LATTE, 8 FL OZ
Tokens:   ['death', 'wish', 'coffee', 'latte']
after token sent:   death wish coffee latte

Original:  ORIGINAL ICED W/ MILK CAPPUCCINO
Tokens:   ['original', 'iced', 'milk', 'cappuccino']
after token sent:   original iced milk cappuccino

Original:  ZERO CALORIE SPORTS DRINK, FRUIT PUNCH
Tokens:   ['zero', 'calorie', 'sport', 'drink', 'fruit', 'punch']
after token sent:   zero calorie sport drink fruit punch

Original: ""BACN,CAND,FL,4PC, Z""
Tokens:   ['bacn', 'cand']
after token sent:   bacn cand

Original: .38 SPECIAL GOOD ON EVERYTHING SEASONING, .38 SPECIAL
Tokens:   ['special', 'good', 'everything', 'seasoning']
after token sent:   special good everything seasonin

In [99]:
# from nltk.tokenize import RegexpTokenizer

# tokenizer = RegexpTokenizer(r'\w+|\d+')

# # text = df.description[480131]
# # tokens = tokenizer.tokenize(text)
# # print(tokens)

# import nltk
# # nltk.download('punkt_tab')
# # nltk.download('averaged_perceptron_tagger_eng')

# from nltk.tokenize import word_tokenize
# from nltk import pos_tag

# text2 = df.description_clean[24]
# # ## 550-553

# tokens2 = tokenizer.tokenize(text2)
# tagged_tokens2 = pos_tag(tokens2)

# print(tagged_tokens2)

# # res = False
# for index, text in enumerate(df.description_clean):
#     tokens = tokenizer.tokenize(text)
#     tagged_tokens = pos_tag(tokens)
#     for w, tag in tagged_tokens:
#         if tag == 'CD' or len(w)<=2:
#             print(f'at index {index} the token was {w}.')
#             break

# '''
# Col = df.description_clean.fillna('-')
# col_str = Col.astype(str)
# All = ''.join(col_str)
# print('joind all of them ', All[10:30])
# ls =[]
# tokens = tokenizer.tokenize(All)
# ls.extend(pos_tag(tokens))
# S = sorted(set(ls))
# print(S)

# ls = []
# S =[]
# for text in df.description:
#     tokens = tokenizer.tokenize(text)
#     ls.extend(pos_tag(tokens))
#     S = sorted(set(ls))

# print(S)
# '''

In [134]:
# description_clean = [preprocess_column(x) for x in df.description]
# combining the description and the category into one list

# combine_with_space = lambda a, b: f"{a} {b}"
for i in range(0,5):
     print(f"{df.description[i]} {df.category[i]}")
     
D = [preprocess_food_entries(f"{desc} {cat}", False) 
     for desc, cat in zip(df['description'], df['category'])]

print(D[0:5])



 ADVANCEPIERRE FLAMEBROILED RIB SHAPED PORK PATTY TOPPED WITH BBQ SAUCE  Meat/Poultry/Other Animals - Prepared/Processed
 ALL NATURAL GLUTEN FREE CHICKEN NUGGETS Frozen Poultry, Chicken & Turkey
 ALL NATURAL ROSEMARY & OLIVE OIL BASMATI RICE, ROSEMARY; OLIVE OIL Flavored Rice Dishes
 ALMOND MILK Plant Based Milk
 ALMOND MILK, ORIGINAL Plant Based Milk
['advancepierre flamebroiled rib shaped pork patty topped bbq sauce meat poultry animal prepared processed', 'natural gluten free chicken nugget frozen poultry turkey', 'natural rosemary olive oil basmati rice flavored dish', 'almond milk plant based', 'almond milk original plant based']


In [ ]:
# df['description_clean'] = description_clean

### VECTORIZATION
# Choose your vectorization method

vectorization_method = 'tfidf'  # Options: 'tfidf', 'count', 'binary'

if vectorization_method == 'tfidf':
    print("Using TF-IDF Vectorization")
    vectorizer = TfidfVectorizer(
        max_features=2000,
        ngram_range=(1, 2),
        min_df=20,
        max_df=0.8,
        stop_words='english'
    )
    X = vectorizer.fit_transform(df['description_clean']).toarray()
    
elif vectorization_method == 'count':
    print("Using Count Vectorization")
    vectorizer = CountVectorizer(
        max_features=2000,
        ngram_range=(1, 2),
        min_df=20,
        max_df=0.8,
        stop_words='english'
    )
    X = vectorizer.fit_transform(df['description_clean']).toarray()
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
else:  # binary
    print("Using Binary Vectorization")
    vectorizer = CountVectorizer(
        max_features=2000,
        binary=True,
        ngram_range=(1, 1),
        min_df=20,
        max_df=0.8,
        stop_words='english'
    )
    X = vectorizer.fit_transform(df['description_clean']).toarray()

print(f"Vectorized shape: {X.shape}")
print(f"Feature dimension: {X.shape[1]}")


In [32]:
# Show top features
feature_names = vectorizer.get_feature_names_out()
print(f"\nTop 20 features: {feature_names[40:50]}")



Top 20 features: ['ale' 'alfredo' 'alfredo sauce' 'almond' 'almond butter' 'almond milk'
 'almondmilk' 'almonds' 'almonds cashews' 'aloe']


In [22]:
# Convert to PyTorch tensors
X_tensor = torch.FloatTensor(X)

# Create DataLoader for batch processing
batch_size = 512
dataset = TensorDataset(X_tensor)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"\nDataLoader created with batch size: {batch_size}")
print(f"Number of batches: {len(dataloader)}")

# Save processed data (optional)
# torch.save(X_tensor, 'X_tensor.pt')
# df.to_csv('cleaned_descriptions.csv', index=False)




Top 20 features: ['10' '10 oz' '100' '100 fruit' '100 grain' '100 juice' '100 natural'
 '100 organic' '100 pure' '100 wheat' '12' '12 oz' '13' '14' '15' '15 oz'
 '16' '16 oz' '18' '18 fat']

DataLoader created with batch size: 512
Number of batches: 938


## PIECE 2: Neural Network Training


In [23]:

# Hyperparameters
input_dim = X.shape[1]
encoding_dim = 50
n_clusters = 20  # Adjust based on expected food categories
n_epochs = 50
learning_rate = 0.001

print(f"Input dimension: {input_dim}")
print(f"Encoding dimension: {encoding_dim}")
print(f"Number of clusters: {n_clusters}")
print(f"Epochs: {n_epochs}")

# Build encoder
encoder = nn.Sequential(
    nn.Linear(input_dim, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, encoding_dim),
    nn.ReLU()
)

# Build decoder
decoder = nn.Sequential(
    nn.Linear(encoding_dim, 64),
    nn.ReLU(),
    nn.Linear(64, 128),
    nn.ReLU(),
    nn.Linear(128, input_dim),
    nn.Sigmoid()
)

# Clustering layer
clustering_layer = nn.Linear(encoding_dim, n_clusters)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(
    list(encoder.parameters()) + 
    list(decoder.parameters()) + 
    list(clustering_layer.parameters()), 
    lr=learning_rate
)

# Training history for plotting
history = {
    'epoch': [],
    'total_loss': [],
    'reconstruction_loss': [],
    'clustering_loss': []
}

print("\nStarting training...")

# ==== TRAINING LOOP ====
for epoch in range(n_epochs):
    epoch_total_loss = 0
    epoch_recon_loss = 0
    epoch_cluster_loss = 0
    
    for batch_idx, (batch_X,) in enumerate(dataloader):
        # Forward pass
        encoded = encoder(batch_X)
        decoded = decoder(encoded)
        cluster_logits = clustering_layer(encoded)
        
        # Reconstruction loss
        reconstruction_loss = criterion(decoded, batch_X)
        
        # Clustering loss (KL divergence with target distribution)
        cluster_probs = torch.softmax(cluster_logits, dim=1)
        target_dist = cluster_probs ** 2 / (cluster_probs.sum(0) + 1e-10)
        target_dist = target_dist / (target_dist.sum(1, keepdim=True) + 1e-10)
        clustering_loss = -(target_dist * torch.log(cluster_probs + 1e-10)).sum(1).mean()
        
        # Total loss
        total_loss = reconstruction_loss + 0.1 * clustering_loss
        
        # Backward pass
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        # Accumulate losses
        epoch_total_loss += total_loss.item()
        epoch_recon_loss += reconstruction_loss.item()
        epoch_cluster_loss += clustering_loss.item()
    
    # Calculate average losses
    avg_total_loss = epoch_total_loss / len(dataloader)
    avg_recon_loss = epoch_recon_loss / len(dataloader)
    avg_cluster_loss = epoch_cluster_loss / len(dataloader)
    
    # Store history
    history['epoch'].append(epoch + 1)
    history['total_loss'].append(avg_total_loss)
    history['reconstruction_loss'].append(avg_recon_loss)
    history['clustering_loss'].append(avg_cluster_loss)
    
    # Print progress
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{n_epochs}] - '
              f'Total Loss: {avg_total_loss:.4f}, '
              f'Recon Loss: {avg_recon_loss:.4f}, '
              f'Cluster Loss: {avg_cluster_loss:.4f}')

print("\n✓ Training complete!")

# Save model (optional)
# torch.save(encoder.state_dict(), 'encoder.pth')
# torch.save(clustering_layer.state_dict(), 'clustering_layer.pth')


Input dimension: 2000
Encoding dimension: 50
Number of clusters: 20
Epochs: 50

Starting training...
Epoch [5/50] - Total Loss: 0.0005, Recon Loss: 0.0005, Cluster Loss: 0.0000
Epoch [10/50] - Total Loss: 0.0005, Recon Loss: 0.0005, Cluster Loss: 0.0000
Epoch [15/50] - Total Loss: 0.0005, Recon Loss: 0.0005, Cluster Loss: 0.0000
Epoch [20/50] - Total Loss: 0.0005, Recon Loss: 0.0005, Cluster Loss: 0.0000
Epoch [25/50] - Total Loss: 0.0005, Recon Loss: 0.0005, Cluster Loss: 0.0000
Epoch [30/50] - Total Loss: 0.0005, Recon Loss: 0.0005, Cluster Loss: 0.0000
Epoch [35/50] - Total Loss: 0.0005, Recon Loss: 0.0005, Cluster Loss: 0.0000
Epoch [40/50] - Total Loss: 0.0005, Recon Loss: 0.0005, Cluster Loss: 0.0000
Epoch [45/50] - Total Loss: 0.0005, Recon Loss: 0.0005, Cluster Loss: 0.0000
Epoch [50/50] - Total Loss: 0.0005, Recon Loss: 0.0005, Cluster Loss: 0.0000

✓ Training complete!


## PIECE 3: Results and Visualization

In [ ]:
encoder.eval()
clustering_layer.eval()

with torch.no_grad():
    encoded = encoder(X_tensor)
    cluster_logits = clustering_layer(encoded)
    cluster_assignments = torch.argmax(cluster_logits, dim=1).numpy()
    cluster_probabilities = torch.softmax(cluster_logits, dim=1).numpy()

# Add to dataframe
df['cluster'] = cluster_assignments
df['cluster_confidence'] = cluster_probabilities.max(axis=1)

print(f"\nCluster distribution:")
print(df['cluster'].value_counts().sort_index())

# ==== VISUALIZATION ====
print("\n=== Creating Visualizations ===")

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

# Create figure with subplots
fig = plt.figure(figsize=(18, 12))

# 1. Loss curves over epochs
ax1 = plt.subplot(2, 3, 1)
plt.plot(history['epoch'], history['total_loss'], label='Total Loss', linewidth=2, color='blue')
plt.plot(history['epoch'], history['reconstruction_loss'], label='Reconstruction Loss', linewidth=2, color='green')
plt.plot(history['epoch'], history['clustering_loss'], label='Clustering Loss', linewidth=2, color='red')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training Loss Over Epochs', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Total loss only (zoomed)
ax2 = plt.subplot(2, 3, 2)
plt.plot(history['epoch'], history['total_loss'], linewidth=2, color='purple', marker='o', markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Total Loss', fontsize=12)
plt.title('Total Loss Progression', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# 3. Cluster distribution (bar chart)
ax3 = plt.subplot(2, 3, 3)
cluster_counts = df['cluster'].value_counts().sort_index()
plt.bar(cluster_counts.index, cluster_counts.values, color='steelblue', edgecolor='black')
plt.xlabel('Cluster ID', fontsize=12)
plt.ylabel('Number of Items', fontsize=12)
plt.title('Distribution of Items Across Clusters', fontsize=14, fontweight='bold')
plt.xticks(range(n_clusters))
plt.grid(True, alpha=0.3, axis='y')

# 4. Cluster confidence distribution
ax4 = plt.subplot(2, 3, 4)
plt.hist(df['cluster_confidence'], bins=50, color='coral', edgecolor='black', alpha=0.7)
plt.xlabel('Confidence Score', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Cluster Assignment Confidence Distribution', fontsize=14, fontweight='bold')
plt.axvline(df['cluster_confidence'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["cluster_confidence"].mean():.3f}')
plt.legend()
plt.grid(True, alpha=0.3)

# 5. Loss reduction percentage
ax5 = plt.subplot(2, 3, 5)
initial_loss = history['total_loss'][0]
final_loss = history['total_loss'][-1]
loss_reduction = [(initial_loss - loss) / initial_loss * 100 for loss in history['total_loss']]
plt.plot(history['epoch'], loss_reduction, linewidth=2, color='green', marker='s', markersize=4)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss Reduction (%)', fontsize=12)
plt.title(f'Loss Reduction: {loss_reduction[-1]:.1f}% improvement', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# 6. Sample results table
ax6 = plt.subplot(2, 3, 6)
ax6.axis('tight')
ax6.axis('off')

# Show sample from each cluster (first 5 clusters)
sample_data = []
for cluster_id in range(min(5, n_clusters)):
    cluster_samples = df[df['cluster'] == cluster_id].head(3)
    for idx, row in cluster_samples.iterrows():
        sample_data.append([
            f"C{cluster_id}",
            row['description'][:40] + "..." if len(row['description']) > 40 else row['description'],
            f"{row['cluster_confidence']:.3f}"
        ])

table = ax6.table(cellText=sample_data,
                  colLabels=['Cluster', 'Description', 'Confidence'],
                  cellLoc='left',
                  loc='center',
                  colWidths=[0.1, 0.7, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 2)
ax6.set_title('Sample Cluster Assignments', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('clustering_results.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:

# ==== DETAILED CLUSTER ANALYSIS ====
print("\n=== Cluster Analysis ===")

for cluster_id in range(n_clusters):
    cluster_data = df[df['cluster'] == cluster_id]
    print(f"\n--- Cluster {cluster_id} ({len(cluster_data)} items) ---")
    print(f"Average confidence: {cluster_data['cluster_confidence'].mean():.3f}")
    print(f"Sample descriptions:")
    for idx, desc in enumerate(cluster_data['description'].head(5)):
        print(f"  {idx+1}. {desc}")

# ==== SUMMARY STATISTICS ====
print("\n=== Summary Statistics ===")
print(f"Total items clustered: {len(df)}")
print(f"Number of clusters: {n_clusters}")
print(f"Average cluster size: {len(df) / n_clusters:.1f}")
print(f"Average confidence: {df['cluster_confidence'].mean():.3f}")
print(f"Final total loss: {history['total_loss'][-1]:.4f}")
print(f"Loss reduction: {loss_reduction[-1]:.1f}%")

# Save results
df.to_csv('clustered_food_descriptions.csv', index=False)
print("\n✓ Results saved to 'clustered_food_descriptions.csv'")
print("✓ Visualization saved to 'clustering_results.png'")
